# Module 4 — PAM50 Subtype Classification

**Biological question:** Can we predict a tumour's PAM50 molecular subtype from its bulk RNA-seq gene expression profile? A classifier with good cross-validated accuracy validates that the expression data carries genuine subtype information — a prerequisite for downstream modelling.

**Method:**
- **Feature engineering:** log₂(counts + 1) normalisation (same as in Module 2).
- **Model:** Multinomial logistic regression with L2 regularisation (C=0.1), after standardising each gene to zero mean / unit variance. `lbfgs` solver handles multi-class natively.
- **Evaluation:** 5-fold stratified cross-validation. Metrics: accuracy, balanced accuracy (accounts for class imbalance), macro F1.
- **Inspection:** Top 20 genes by logistic regression coefficient magnitude reveal which genes are most discriminative per subtype.

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
import pickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.constants import InterimFile
from src.paths import interim_file, PROCESSED_DIR
from src.features.differential_expression import log2_normalize
from src.models.classifier import build_pipeline, cross_validate_model, train, confusion_matrix_df
from src.visualization.classification import plot_confusion_matrix

logging.basicConfig(level=logging.INFO, format="%(message)s")

## Load and Normalise Data

In [ ]:
X_raw = pd.read_csv(interim_file(InterimFile.X), index_col=0)
labels = pd.read_csv(interim_file(InterimFile.LABELS), index_col=0).squeeze()

X = log2_normalize(X_raw)
print(f"X (log2): {X.shape[0]} samples × {X.shape[1]} genes")
print(labels.value_counts())

## 5-Fold Cross-Validation

In [ ]:
pipeline = build_pipeline()
cv_results = cross_validate_model(X=X, y=labels, pipeline=pipeline, cv=5)

cv_summary = pd.DataFrame(
    {
        "fold": range(1, 6),
        "accuracy": cv_results["accuracy"],
        "balanced_accuracy": cv_results["balanced_accuracy"],
        "f1_macro": cv_results["f1_macro"],
    }
)
cv_summary.loc["mean"] = cv_summary.mean()
cv_summary.loc["std"] = cv_summary.iloc[:5].std()
cv_summary.style.format("{:.3f}", subset=["accuracy", "balanced_accuracy", "f1_macro"]).highlight_max(axis=0)

## Train on Full Dataset

In [ ]:
fitted = train(X=X, y=labels, pipeline=build_pipeline())
y_pred = fitted.predict(X.values)

## Confusion Matrix

Rows = true subtype; columns = predicted subtype. Numbers show counts and the percentage of that true class predicted as each subtype. A perfect classifier has 100% on the diagonal.

In [ ]:
label_order = sorted(labels.unique())
cm_df = confusion_matrix_df(y_true=labels, y_pred=y_pred, labels=label_order)

fig, ax = plt.subplots(figsize=(7, 6))
plot_confusion_matrix(cm_df=cm_df, ax=ax)
plt.tight_layout()
plt.show()

## Top 20 Predictive Genes by Logistic Regression Coefficient Magnitude

Each subtype has a coefficient vector over all genes. Genes with large |coefficient| push a sample strongly toward or away from that class. These tend to overlap with the DE markers found in Module 2.

In [ ]:
lr = fitted.named_steps["lr"]
classes = lr.classes_
coef_df = pd.DataFrame(lr.coef_, index=classes, columns=X.columns)

top_genes_by_subtype = {}
for subtype in classes:
    top = coef_df.loc[subtype].abs().nlargest(20)
    top_genes_by_subtype[subtype] = top

fig, axes = plt.subplots(1, len(classes), figsize=(4 * len(classes), 5), sharey=False)
for ax, subtype in zip(axes, classes):
    genes = top_genes_by_subtype[subtype]
    colors = ["#d62728" if coef_df.loc[subtype, g] > 0 else "#1f77b4" for g in genes.index]
    ax.barh(range(len(genes)), genes.values, color=colors)
    ax.set_yticks(range(len(genes)))
    ax.set_yticklabels(genes.index, fontsize=7)
    ax.set_title(subtype, fontsize=10, fontweight="bold")
    ax.set_xlabel("|coefficient|")

plt.suptitle("Top 20 genes by |LR coefficient| per subtype", fontsize=12)
plt.tight_layout()
plt.show()

## Save Trained Model

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out = PROCESSED_DIR / "classifier.pkl"
with open(out, "wb") as f:
    pickle.dump(fitted, f)
print(f"Saved {out.name}")